In [13]:
import os
import json
from langchain_community.document_loaders import DirectoryLoader, JSONLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceInferenceAPIEmbeddings
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from dotenv import load_dotenv


In [14]:
loader = DirectoryLoader(
        './corpus_data',
        glob='**/*.json',
        loader_cls=JSONLoader,
        loader_kwargs={'jq_schema': '.content', 'text_content': True}
    )

print("📦 Loading documents...")
docs = loader.load()

📦 Loading documents...


In [15]:
text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=700,
        chunk_overlap=100,
        separators=["\n\n", "\n", ".", " ", ""]
    )
chunks = text_splitter.split_documents(docs)

In [16]:
embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7874.55it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [17]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db",
)

In [18]:
stored = vector_store.from_documents(chunks, embeddings)

In [19]:
stored.similarity_search("How to make Katzu curry", k=5)

[Document(id='d67ff324-72ef-447e-b6dd-e33d338abaf1', metadata={'source': '/Users/vedant/Documents/master_chef/corpus_data/japanese_cuisine.json', 'seq_num': 1}, page_content='Curry was introduced by Anglo-Indian officers of the Royal Navy from India who brought curry powder to Japan in the Meiji era.[99] The Imperial Japanese Navy adopted curry to prevent beriberi. Overtime it was reinvented and adapted to suit Japanese tastes that it became uniquely Japanese.[99] It is consumed so much that it is considered a national dish.[84] Many recipes are on the menu of the JMSDF.[100] A variety of vegetables and meats are used to make Japanese curry, usually vegetables like onions, carrots, and potatoes. The types of meat used are beef, pork, and chicken'),
 Document(id='d02ce6bd-09e8-4a90-8588-f7a80b3060d8', metadata={'source': '/Users/vedant/Documents/master_chef/corpus_data/japanese_cuisine.json', 'seq_num': 1}, page_content='Curry was introduced by Anglo-Indian officers of the Royal Navy fr

In [20]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [21]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

Loading weights: 100%|██████████| 290/290 [00:02<00:00, 128.66it/s]


In [28]:
def generate_culinary_answer(query):
    # 3. Retrieve top 5 chunks (B.1.3 constraint)
    docs = stored.similarity_search(query, k=5)
    context = "\n\n".join([doc.page_content for doc in docs])

    # 4. Construct the Prompt (B.1.4 strategy)
    # We use a strict persona to keep the small model on track
    messages = [
        {"role": "system", "content": "You are a specialist East Asian Master Chef. Use the provided context to answer the user's question accurately and verbosely if there are any explanations. If the answer is not in the context, say you don't know based on the current records. Be concise. make sure you include the source of the information"},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"}
    ]
    
    # Apply ChatML template
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # 5. Generate
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512,
        temperature=0.1 # Low temperature for factual accuracy
    )
    
    # Remove the prompt from the output
    response = tokenizer.batch_decode(
        [out[len(in_ids):] for in_ids, out in zip(model_inputs.input_ids, generated_ids)],
        skip_special_tokens=True
    )[0]
    
    return response

In [31]:
generate_culinary_answer("What foods were introduced to Korea through the Mongol invasion of Goryeo?")

'Based on the information provided, some traditional foods found today in Korea have their origins during the Mongol invasion of Goryeo in the 13th century. These include:\n\n1. Dumplings - Grilled meat dishes\n2. Grilled meat dishes - Noodle dishes\n3. Noodles - Grilled meat dishes\n4. Seasonings like black pepper - All have their roots in this period.\n5. Yuki-po - A type of po, used for seafood\n6. Eopo - Dried beef\n\nThese foods reflect the agricultural practices and dietary preferences of the time, which included rearing livestock and consuming meat in various forms.'